In [1]:
# !pip install -U torch==2.5.1 'accelerate>=0.26.0' bitsandbytes qwen_vl_utils torchvision transformers huggingface_hub fbgemm-gpu
# !pip install git+https://github.com/huggingface/transformers.git qwen_vl_utils torchvision accelerate>=0.26.0
# pip install git+https://github.com/huggingface/transformers accelerate bitsandbytes qwen_vl_utils torchvision
!pip install 'torch>=2.6' 'transformers<4.54.0' 'accelerate>=0.26.0' bitsandbytes qwen_vl_utils torchvision

  Using cached torch-2.8.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (30 kB)
  Using cached transformers-4.53.3-py3-none-any.whl.metadata (40 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.8.93-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_runtime_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_cupti_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cudnn_cu12-9.10.2.21-py3-none-manylinux_2_27_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_cublas_cu12-12.8.4.1-py3-none-manylinux_2_27_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cufft_cu12-11.3.3.83-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_curand_cu12-10.3.9.90-py3-none-manylinux_2_27_x86_64.whl.metadata (1.7 kB)
  Using cached nvid

In [2]:
!nvidia-smi

Tue Oct 14 22:40:36 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.127.08             Driver Version: 550.127.08     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          On  |   00000000:54:00.0 Off |                    0 |
| N/A   29C    P0             73W /  700W |       1MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [1]:
from simulations_core import *

In [2]:
# 4B, 12B, 27B

model_label = 'gemma_27B'

model, processor = import_gemma_27B()

generation_kwargs = {
    "max_new_tokens": 256,
    "do_sample": False,
    # "top_p": 0.9,
    # "temperature": 0.7,
    "return_dict_in_generate": True, 
    "output_scores": True,               # <--- Aggiunto
    "use_cache" : False  # 👈 aggiungi questa riga
}

Loading checkpoint shards:   0%|          | 0/12 [00:00<?, ?it/s]

## CREATE GENERAL POOL OF IMAGES

In [ ]:
import os
import random
import shutil
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")
from transformers.utils import logging
logging.set_verbosity_error()

# Parametri fissi
reference_length = 300
base_width = 300
base_height = 600
spacing = 120
scale = 1.5
dpi = 150
labels = ['A', 'B']
bounds = [1, 580]
difference =200
n_figs = 100

# Cartella di output
output_dir = os.path.join("..", "images", f"images_lines_{model_label}")
os.makedirs(output_dir, exist_ok=True)

# --- LOOP DI GENERAZIONE ---
fig_index = 0
attempts = 0
pbar = tqdm(total=n_figs, desc="Generazione immagini valide")

while fig_index < n_figs:
    # Scegli posizione della linea corretta
    position = random.randint(0, 1)
    lines = [0, 0]
    lines[position] = reference_length

    # Genera una linea più corta e una più lunga
    if random.random() < 0.5:
        random_length = random.randint(bounds[0], reference_length - difference)
    else:
        random_length = random.randint(reference_length + difference, bounds[1])
    
    # Trova le due posizioni libere
    free_position = [i for i in range(2) if i != position]
    lines[free_position[0]] = random_length

    # Crea immagine
    temp_save_path = os.path.join(output_dir, f"temp_image.png")
    generate_asch_image_two_lines(lines, position, base_width, base_height, spacing, scale, temp_save_path, dpi, labels)
    
    # Costruisci prompt
    prompt = (
        f"In the image, there are three vertical lines labeled {labels[0]}, REFERENCE LINE, and {labels[1]}.\n"
        f"Which of the lines, {labels[0]} or {labels[1]}, is the same length as the REFERENCE LINE?\n"
        f"Provide only the final answer, either {labels[0]} or {labels[1]}, without generating anything else."
    )

    # Chiama il modello
    # try:
    prob_labels, output_scores = single_query_gemma(prompt, temp_save_path, labels, model, processor, generation_kwargs, print_flag=False)

    correct_answer = labels[position]
    prob_correct = prob_labels.get(correct_answer, 0.0)
    # print(correct_answer)
    # print(prob_labels)

    # Verifica soglia
    if prob_correct >= 0.95:
        fig_index += 1
        pbar.update(1)
        final_save_path = os.path.join(output_dir, f"image_lines_{fig_index}_{labels[position]}_p={prob_correct:.2f}.png")
        os.rename(temp_save_path, final_save_path)
    else:
        # Immagine scartata
        os.remove(temp_save_path)
        # print('scartata')

    attempts += 1

pbar.close()
print(f"\n✅ Completato: {n_figs} immagini generate correttamente su {attempts} tentativi.")

## CREATE IMAGES FOR TASK DIFFICULTY AND PERFORMANCE

In [ ]:
import os
import random
from tqdm import tqdm

# === Parametri fissi immagine e generazione ===
reference_length = 300
base_width = 300
base_height = 600
spacing = 120
scale = 1.5
dpi = 150
labels = ['A', 'B']

# lunghezze consentite per le linee "distrattori"
bounds = [1, 580]

# soglia di accettazione
prob_threshold = 0.95

# numero immagini per ciascuna difference
n_images = 50

# differenze da esplorare (modifica a piacere)
differences = list(range(50, 251, 20))  # 10, 20, ..., 200

# cartella di output (unica, come nel caso "colori")
output_dir = os.path.join("..", "images", f"images_lines_perplexity_diff_{model_label}")
os.makedirs(output_dir, exist_ok=True)

for difference in differences:
    print(f"\n=== difference = {difference} ===")

    fig_index = 0
    attempts = 0
    pbar = tqdm(total=n_images, desc=f"Generazione immagini valide (diff={difference})")

    # # Pre-check rapido: esiste almeno un lato campionabile?
    # # lato "più corto": [bounds[0], reference_length - difference]
    # low_min = bounds[0]
    # low_max = reference_length - difference
    # # lato "più lungo": [reference_length + difference, bounds[1]]
    # high_min = reference_length + difference
    # high_max = bounds[1]

    low_min = max(bounds[0], reference_length - difference)
    low_max = reference_length - 1

    high_min = reference_length + 1
    high_max = min(bounds[1], reference_length + difference)

    if low_min > low_max and high_min > high_max:
        pbar.close()
        print(f"⚠️ Nessun range valido per difference={difference} con bounds={bounds}. Salto.")
        continue

    while fig_index < n_images:
        # posizione della linea corretta (REFERENCE)
        position = random.randint(0, 1)
        lines = [0, 0]
        lines[position] = reference_length
        correct_answer = labels[position]

        # Scegli quale lato (più corto o più lungo) in base a quelli effettivamente validi
        eligible_sides = []
        if low_min <= low_max:
            eligible_sides.append("short")
        if high_min <= high_max:
            eligible_sides.append("long")

        side = random.choice(eligible_sides)
        if side == "short":
            random_length = random.randint(low_min, low_max)
        else:  # "long"
            random_length = random.randint(high_min, high_max)

        # posizione dell'altra linea
        other_pos = 1 - position
        lines[other_pos] = random_length

        # Crea immagine temporanea
        temp_path = os.path.join(output_dir, "temp.png")
        generate_asch_image_two_lines(
            lines,
            position,
            base_width=base_width,
            base_height=base_height,
            spacing=spacing,
            scale=scale,
            save_path=temp_path,
            dpi=dpi,
            labels=labels
        )

        # Prompt per il modello (stile analogo al tuo)
        prompt = (
            f"In the image, there are three vertical lines labeled {labels[0]}, REFERENCE LINE, and {labels[1]}.\n"
            f"Which of the lines, {labels[0]} or {labels[1]}, is the same length as the REFERENCE LINE?\n"
            f"Provide only the final answer, either {labels[0]} or {labels[1]}, without generating anything else."
        )

        # Query al modello
        prob_labels, output_scores = single_query_gemma(
            prompt, temp_path, labels, model, processor, generation_kwargs, print_flag=False
        )
        prob_correct = prob_labels.get(correct_answer, 0.0)
        logit = output_scores.get(correct_answer, 0.0)

        # Nome file con metadata (includo difference)
        image_name = f"lines_{fig_index}_{correct_answer}_p={prob_correct:.2f}_logit={logit:.2f}_diff={difference}_other={random_length}.png"
        image_path = os.path.join(output_dir, image_name)

        # Accettazione o scarto
        if prob_correct >= prob_threshold:
            fig_index += 1
            pbar.update(1)
            os.rename(temp_path, image_path)
        else:
            # scarta
            try:
                os.remove(temp_path)
            except FileNotFoundError:
                pass

        attempts += 1

    pbar.close()
    print(f"✅ Completato diff={difference}: {n_images} immagini su {attempts} tentativi.")